# Medulloblastoma MRI — Signal and Image Processing

**10 images × 10 operations**, based on SIP practicals 2–8 and the original MATLAB pipeline. The segmentation outputs are illustrative and **not diagnostic tumour masks**.

## How to run in Google Colab

1. Open [Google Colab](https://colab.research.google.com/) and use **File → Upload notebook** to select this `.ipynb` from the downloaded ZIP (extract it locally first).
2. Run the first code cell to upload the **original shared ZIP** (`Medulloblastoma_SIP_Colab.zip`).
3. Run the install cell and then run the main pipeline cell. Images appear **inline**; outputs are saved to `results/`.

**If running locally in Jupyter:** extract this ZIP, open the notebook inside its folder, skip the upload cell, and run the install and pipeline cells.

In [ ]:
# COLAB ONLY: upload the complete Medulloblastoma_SIP_Colab.zip file
# In local Jupyter, skip this cell.
try:
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    for archive_name, contents in uploaded.items():
        if archive_name.lower().endswith('.zip'):
            with zipfile.ZipFile(archive_name) as z:
                z.extractall('.')
            print('Dataset ZIP extracted.')
except ImportError:
    print('Not in Colab; use the images in the extracted MB_10_images folder.')

In [ ]:
%pip -q install numpy opencv-python matplotlib pandas scipy

## Complete Python pipeline — all 10 operations
Run the cell below. It will display figures inline and save PNGs and `results/metrics.csv`. Each MRI produces **six comparison figures**, including a separate histogram figure.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import convolve
from IPython.display import display

# Ten operations based on the SIP practicals and original MATLAB pipeline.
# Not intended for diagnosis or validated tumour segmentation.

INPUT_DIR = next((p for p in [Path('MB_10_images'), Path('Medulloblastoma_SIP_Colab/MB_10_images'), Path('/content/MB_10_images'), Path('/content/Medulloblastoma_SIP_Colab/MB_10_images')] if p.is_dir()), None)
if INPUT_DIR is None:
    raise FileNotFoundError('MB_10_images was not found. Run the upload/unzip cell first.')
OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)
image_files = sorted(p for p in INPUT_DIR.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.tif','.tiff'} and p.is_file())[:10]
if not image_files:
    raise ValueError(f'No supported MRI images in {INPUT_DIR}')
print(f'Found {len(image_files)} MRI images in {INPUT_DIR}')

def butterworth(image, cutoff, order=2, kind='low'):
    rows, cols = image.shape
    F = np.fft.fftshift(np.fft.fft2(image))
    y, x = np.meshgrid(np.arange(rows)-rows//2, np.arange(cols)-cols//2, indexing='ij')
    radius = np.hypot(x,y)
    if kind == 'low':
        H = 1 / (1 + (radius/cutoff)**(2*order))
    else:
        H = 1 / (1 + (cutoff/np.maximum(radius,1e-12))**(2*order))
    result = np.real(np.fft.ifft2(np.fft.ifftshift(F*H)))
    return result, np.log1p(np.abs(F))

def edges(image, kx, ky=None):
    dx = convolve(image, kx, mode='nearest')
    response = np.hypot(dx, convolve(image,ky,mode='nearest')) if ky is not None else abs(dx)
    return (response > response.mean()+2*response.std()).astype('uint8')*255

def component_count(binary):
    count,_ = cv2.connectedComponents((binary>0).astype('uint8'), connectivity=8)
    return count-1

def psnr(reference, processed):
    mse = np.mean((reference.astype(float)-processed.astype(float))**2)
    return float('inf') if mse == 0 else 10*np.log10(255**2/mse)

def show_grid(images, titles, heading, filename, cols=4):
    rows = int(np.ceil(len(images)/cols))
    fig, ax = plt.subplots(rows,cols,figsize=(4*cols,3.5*rows),squeeze=False)
    for a, pic, ttl in zip(ax.flat,images,titles):
        a.imshow(pic,cmap='gray')
        a.set_title(ttl)
        a.axis('off')
    for a in list(ax.flat)[len(images):]: a.axis('off')
    fig.suptitle(heading,fontsize=15)
    fig.tight_layout()
    fig.savefig(filename,dpi=140,bbox_inches='tight')
    plt.show()
    plt.close(fig)

rng=np.random.default_rng(1)
metrics=[]
for idx, path in enumerate(image_files,1):
    print(f'Processing {idx}/{len(image_files)}: {path.name}')
    dest=OUTPUT_DIR/path.stem
    dest.mkdir(exist_ok=True)
    source=cv2.imread(str(path),cv2.IMREAD_UNCHANGED)
    if source is None:
        print('Skipped unreadable file:',path)
        continue

    # 1. Grayscale conversion (Practical 3)
    if source.ndim==2: gray=source
    elif source.shape[2]==4: gray=cv2.cvtColor(source,cv2.COLOR_BGRA2GRAY)
    else: gray=cv2.cvtColor(source,cv2.COLOR_BGR2GRAY)
    if gray.dtype!=np.uint8: gray=cv2.normalize(gray,None,0,255,cv2.NORM_MINMAX).astype('uint8')
    norm=gray.astype(float)/255

    # 2. Contrast stretching (Practical 4)
    lo,hi=np.percentile(gray,[1,99])
    contrast=np.clip((gray.astype(float)-lo)*255/(hi-lo),0,255).astype('uint8') if hi>lo else gray.copy()

    # 3. Gamma transforms (Practical 4)
    gamma05=np.clip(255*(contrast/255)**0.5,0,255).astype('uint8')
    gamma15=np.clip(255*(contrast/255)**1.5,0,255).astype('uint8')

    # 4. Histogram equalization (Practical 5)
    equalized=cv2.equalizeHist(contrast)
    show_grid([gray,contrast,gamma05,gamma15,equalized],['1. Grayscale','2. Contrast stretch','3. Gamma 0.5','3. Gamma 1.5','4. Histogram equalization'],f'{path.stem}: operations 1-4',dest/'operations_1_to_4.png')
    fig,ax=plt.subplots(1,2,figsize=(10,3.5))
    ax[0].hist(gray.ravel(),bins=256,range=(0,256)); ax[0].set_title('Original histogram'); ax[0].set_xlabel('Gray level')
    ax[1].hist(equalized.ravel(),bins=256,range=(0,256)); ax[1].set_title('Equalized histogram'); ax[1].set_xlabel('Gray level')
    fig.tight_layout(); fig.savefig(dest/'histograms.png',dpi=140); plt.show(); plt.close(fig)

    # 5. Median filtering on 5% salt-and-pepper noise (Practical 6)
    n=rng.random(gray.shape); noisy=gray.copy(); noisy[n<0.025]=0; noisy[n>0.975]=255
    mean3=cv2.blur(noisy,(3,3)); mean5=cv2.blur(noisy,(5,5))
    median3=cv2.medianBlur(noisy,3); median5=cv2.medianBlur(noisy,5)
    show_grid([gray,noisy,mean3,mean5,median3,median5],['Original','Salt & pepper','Mean 3x3','Mean 5x5','5. Median 3x3','5. Median 5x5'],f'{path.stem}: operation 5',dest/'operation_5_median.png')

    # 6. Butterworth low/high pass using FFT (Practical 2 + MATLAB reference)
    noisy_gauss=np.clip(norm+rng.normal(0,np.sqrt(0.005),gray.shape),0,1)
    low,spectrum=butterworth(noisy_gauss,50,2,'low'); high,_=butterworth(norm,30,2,'high')
    show_grid([norm,noisy_gauss,spectrum,np.clip(low,0,1),high,np.abs(high)],['Original','Gaussian noise','FFT spectrum','6. Butterworth LPF','6. Butterworth HPF','HPF magnitude'],f'{path.stem}: operation 6',dest/'operation_6_butterworth.png')

    # 7. Edge detection (Practical 7 + MATLAB reference)
    smoothed=cv2.medianBlur(contrast,3); S=smoothed.astype(float)/255
    robx=np.array([[1,0],[0,-1]]); roby=np.array([[0,1],[-1,0]])
    prex=np.array([[-1,0,1],[-1,0,1],[-1,0,1]])
    sobx=np.array([[-1,0,1],[-2,0,2],[-1,0,1]])
    lap=np.array([[0,1,0],[1,-4,1],[0,1,0]])
    roberts=edges(S,robx,roby); prewitt=edges(S,prex,prex.T)
    sobel=edges(S,sobx,sobx.T); laplacian=edges(S,lap)
    show_grid([gray,roberts,prewitt,sobel,laplacian],['Original','Roberts','Prewitt','7. Sobel','Laplacian'],f'{path.stem}: operation 7',dest/'operation_7_edges.png')

    # 8. Otsu threshold segmentation (MATLAB reference)
    threshold,binary=cv2.threshold(smoothed,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)

    # 9. Erosion/dilation (Practical 8)
    disk=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7))
    square=cv2.getStructuringElement(cv2.MORPH_RECT,(7,7))
    eroded=cv2.erode(binary,disk); dilated=cv2.dilate(binary,square)

    # 10. Opening/closing (MATLAB reference)
    large=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    opened=cv2.morphologyEx(binary,cv2.MORPH_OPEN,large)
    closed=cv2.morphologyEx(binary,cv2.MORPH_CLOSE,large)
    count,labels,stats,_=cv2.connectedComponentsWithStats((opened>0).astype('uint8'),connectivity=8)
    roi=np.zeros_like(binary)
    if count>1:
        largest=1+np.argmax(stats[1:,cv2.CC_STAT_AREA])
        roi[labels==largest]=255
        roi=cv2.morphologyEx(roi,cv2.MORPH_CLOSE,large)
    show_grid([gray,binary,eroded,dilated,opened,closed,roi],['Original','8. Otsu','9. Erosion','9. Dilation','10. Opening','10. Closing','Largest foreground region'],f'{path.stem}: operations 8-10',dest/'operations_8_to_10.png')

    metrics.append({'Image':path.name,'Mean_Gray':float(gray.mean()),'Std_Gray':float(gray.std()),
       'Mean_Contrast':float(contrast.mean()),'Mean_Equalized':float(equalized.mean()),
       'PSNR_SaltPepper':psnr(gray,noisy),'PSNR_Mean3':psnr(gray,mean3),'PSNR_Median3':psnr(gray,median3),
       'PSNR_Median5':psnr(gray,median5),'PSNR_ButterworthLPF':psnr(gray,np.uint8(np.round(np.clip(low,0,1)*255))),
       'Otsu_Threshold':float(threshold),'Foreground_Percent':100*float(np.mean(binary>0)),
       'Roberts_Edge_Percent':100*float(np.mean(roberts>0)),'Prewitt_Edge_Percent':100*float(np.mean(prewitt>0)),
       'Sobel_Edge_Percent':100*float(np.mean(sobel>0)),'Laplacian_Edge_Percent':100*float(np.mean(laplacian>0)),
       'Components_Otsu':component_count(binary),'Components_Eroded':component_count(eroded),
       'Components_Dilated':component_count(dilated),'Components_Opened':component_count(opened),
       'Components_Closed':component_count(closed),'ROI_Percent':100*float(np.mean(roi>0))})
    print('Saved results for',path.name)

if not metrics: raise RuntimeError('No images could be processed')
df=pd.DataFrame(metrics)
df.to_csv(OUTPUT_DIR/'metrics.csv',index=False)
print('\nCompleted:',len(df),'MRI images; 10 operations each')
print('Saved figures and metrics to',OUTPUT_DIR.resolve())
display(df.round(2))
